![Week 2 Day 3](../assets/banners/w2d3.png)

# Week 2 Day 3 - Vector Stores: Giving GradeWise a Memory

Yesterday you turned Emma's Unit 1 notes into clean, structure-aware chunks, each one labelled with its source, page, unit and section. Across all four PDFs, that's a few dozen chunks today, and it could be tens of thousands for a whole department next year.

Today we give those chunks a home: a **vector database**. You'll see how vector search stays fast at scale (HNSW), build a persistent **Chroma** store with one collection per course, search it with **metadata filters**, and finally make ingestion **idempotent**, so that running it a second time adds nothing and costs nothing.

![Your 6-week roadmap](../assets/roadmap/week2.png)

### 🎯 By the end of this lab you will
1. 🕸️ Explain why vector databases use approximate search (HNSW), and pick a distance metric
2. 🗄️ Build a persistent Chroma store, with a collection per course
3. 🔎 Filter searches by metadata: "only Unit 3", "only pages 3 and up of Unit 2"
4. ♻️ Run GradeWise's idempotent `ingest()` twice and watch the second run add **0** chunks

### 🎬 Videos for this lab

| # | Lecture |
|:-:|---|
| 1 | Day 3 - Vector Databases: HNSW, Distance Metrics and Metadata Filters |
| 2 | Day 3 - A Persistent Chroma Store with Collections per Course |
| 3 | Day 3 - Idempotent Ingestion: Hashing, De-duplication, Re-indexing |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Everything in this lab runs <b>offline</b>, in a <b>temporary folder</b>, with <code>DeterministicFakeEmbedding</code>. Fake embeddings make every <i>similarity</i> result meaningless (random), but everything else is 100% real: storage, persistence, collections, metadata filters, hashing and idempotency. Your real knowledge base, built with <code>gradewise.models.embeddings()</code>, lives in <code>output/vector_db</code>, and nothing here touches it.</span>
        </td>
    </tr>
</table>

💡 **Fast & cheap run.** Two lines in `.env` decide what this notebook costs. `MODEL_MODE=free` runs on OpenRouter's free models for $0 (same `OPENROUTER_API_KEY`, no credits, but only about 50 requests a day), and `DATA_SIZE=small` uses a small slice of the data (2 PDFs · Q1–Q3 · 2 students) when you just want to see the APIs work. Restart the kernel after editing `.env`; the next cell prints what you're running.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 🔑 cheap and free both use one key, OPENROUTER_API_KEY (from openrouter.ai/keys); custom may need others
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"ℹ️ {key_name} not set. That's OK for most of this lab: the offline cells use fakes.")
        print("   The few cells that call a real model will tell you so. Setup: Part 5 of the setup guide.")

print(settings.describe_models())
print(datasets.describe())

In [ ]:
import hashlib
import shutil
import tempfile
import time
from pathlib import Path

import numpy as np
from langchain_core.embeddings import DeterministicFakeEmbedding

from gradewise.config import DATA_DIR
from gradewise.ingestion import chunk_documents, get_vectorstore, load_pdf

KB = DATA_DIR / "knowledge-base"

# ⚠️ Offline demo: fake embeddings, temp folder. Real ones: gradewise.models.embeddings()
fake_embeddings = DeterministicFakeEmbedding(size=64)
TMP = Path(tempfile.mkdtemp(prefix="gradewise_w2d3_"))
print(f"Today's throw-away vector stores live in:\n  {TMP}")

## Part 1: Vector databases, HNSW and distance metrics

On Day 1, retrieval was "compare the question's vector with every stored vector, keep the closest". That's called **brute-force** (or *exact*) search, and its cost grows linearly with the number of vectors. Let's feel that with numpy:

In [ ]:
rng = np.random.default_rng(0)
DIM = 384  # 💡 a small embedding model's size; keeps this demo's memory under ~300 MB
query = rng.normal(size=DIM).astype(np.float32)

for n in (10_000, 50_000, 200_000):
    vectors = rng.normal(size=(n, DIM)).astype(np.float32)
    best = float("inf")
    for _ in range(3):  # 💡 best of 3 runs, so a busy laptop doesn't skew the numbers
        t0 = time.perf_counter()
        top5 = np.argpartition(vectors @ query, -5)[-5:]
        best = min(best, time.perf_counter() - t0)
    print(f"{n:>8,} vectors: brute force took {best * 1000:7.1f} ms per query")
del vectors

Fine for GradeWise's few dozen chunks, but at 10 million vectors and many queries per second it's far too slow. So vector databases use **approximate nearest neighbour (ANN)** indexes. The most popular, used by Chroma, is **HNSW** (*Hierarchical Navigable Small World*):

- Every vector is a node in a graph, linked to its nearest neighbours.
- There are several **layers**: the top layer has a few nodes with long-range links (like motorways); lower layers have more nodes and shorter links (like local streets). The bottom layer has every node.
- A search starts at the top, greedily hops towards the query, then drops down a layer and repeats, until it walks the local streets at the bottom.

![Diagram](../assets/diagrams/8a69710c20c6.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart TB
    subgraph L2["Layer 2 · a few nodes, long links"]
        A2((A)) --- F2((F))
    end
    subgraph L1["Layer 1 · more nodes"]
        A1((A)) --- C1((C)) --- F1((F)) --- H1((H))
    end
    subgraph L0["Layer 0 · every node, short links"]
        A0((A)) --- B0((B)) --- C0((C)) --- D0((D)) --- E0((E)) --- F0((F)) --- G0((G)) --- H0((H))
    end
    A2 -.drop down.-> A1
    F2 -.drop down.-> F1
    F1 -.drop down.-> F0
    Q[🔎 query] -.enters at the top.-> A2
```

</details>


The result: search cost grows roughly with the **logarithm** of the number of vectors, at the price of *occasionally* missing the true nearest neighbour. For RAG that trade is excellent.

### Which distance?

| Chroma `hnsw:space` | Measures | Note |
|---|---|---|
| `l2` (Chroma's default) | Squared Euclidean distance | Affected by vector length |
| `ip` | Inner (dot) product | Fast; equals cosine only for unit-length vectors |
| `cosine` | 1 − cosine similarity (0 = same direction, 2 = opposite) | ⭐ The safe choice for text: length doesn't matter |

GradeWise sets `collection_metadata={"hnsw:space": "cosine"}` in `get_vectorstore()`. 🔁 Remember Day 1: cosine ignores length, so a long page and a short sentence on the same topic still match.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">The distance metric is fixed <b>when a collection is created</b>. Changing <code>hnsw:space</code> later does nothing to an existing collection: you must delete it and re-ingest. The same goes for the embedding model. Mixing vectors from two models (or a fake and a real one!) in one collection gives garbage, which is why the offline evaluation on Day 5 writes to its own folder.</span>
        </td>
    </tr>
</table>

## Part 2: A persistent Chroma store with collections per course

Chroma is an open-source vector database that runs inside your Python process and saves to a folder: no server, no account. LangChain's `Chroma` class wraps it, and GradeWise's `get_vectorstore()` wraps that, with our defaults (folder, cosine, embedding model).

First, the chunks: the knowledge-base PDFs (all four, or two with `DATA_SIZE=small`), loaded and chunked exactly like yesterday.

In [ ]:
pages = [page for pdf in datasets.kb_files(KB) for page in load_pdf(pdf)]  # 💡 DATA_SIZE picks the PDFs
chunks = chunk_documents(pages)
print(f"{len(pages)} pages → {len(chunks)} chunks from {len({c.metadata['source'] for c in chunks})} PDFs")

In [ ]:
store = get_vectorstore("cs201", embeddings=fake_embeddings, persist_dir=TMP / "store")

# 💡 Stable IDs: "<file>#<chunk number>". Adding the same ID twice overwrites, never duplicates.
ids = [f"{c.metadata['source']}#{c.metadata['chunk_index']}" for c in chunks]
store.add_documents(chunks, ids=ids)
print(f"✅ Stored {store._collection.count()} chunks in collection '{store._collection.name}'")
print("   distance metric:", store._collection.metadata)

What actually got stored? For every chunk: an **ID**, the **text**, the **metadata** and the **embedding** vector.

In [ ]:
record = store.get(ids=["unit1_operating_systems.pdf#15"], include=["documents", "metadatas", "embeddings"])
print("id       :", record["ids"][0])
print("metadata :", record["metadatas"][0])
print("text     :", record["documents"][0][:120].replace("\n", " "), "…")
print("embedding:", np.round(record["embeddings"][0][:6], 3), f"… ({len(record['embeddings'][0])} numbers)")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Next we run a similarity search for <i>"What are the four necessary conditions for deadlock?"</i>. With <b>fake</b> embeddings, predict: will the top result be from section 1.6 Deadlocks? Run it, then run it again.</span>
        </td>
    </tr>
</table>

In [ ]:
results = store.similarity_search_with_score("What are the four necessary conditions for deadlock?", k=4)
for doc, distance in results:
    print(f"distance {distance:.3f}  {doc.metadata['source']:<30} §{doc.metadata['section']}")

Random sections, but the *same* random sections every time you run it: that's `DeterministicFakeEmbedding` in a nutshell. The plumbing works; the meaning is missing. With `gradewise.models.embeddings()` the top hits would be the deadlock chunks. (On Day 4 you'll meet BM25, which finds them even offline.)

### Persistence

The store lives on disk, so a brand-new Python object (or tomorrow's notebook, or the GradeWise app) can open it again without re-embedding anything:

In [ ]:
reopened = get_vectorstore("cs201", embeddings=fake_embeddings, persist_dir=TMP / "store")
print(f"Reopened from disk: {reopened._collection.count()} chunks, no embedding calls needed")
print("Files on disk:", sorted(p.name for p in (TMP / "store").iterdir()))

### One collection per course

A **collection** is a separate named index inside the same database, like a table in SQL. GradeWise uses one per course: `cs201` for Emma's Systems Fundamentals. If Northbridge adds another course next term, it gets its own collection, and a CS201 question can never be answered from the other course's notes.

Let's pretend there's a second course whose knowledge base is just the handbook:

In [ ]:
handbook_chunks = [c for c in chunks if c.metadata["source"] == "cs201_course_handbook.pdf"]
other_course = get_vectorstore("cs999_demo", embeddings=fake_embeddings, persist_dir=TMP / "store")
other_course.add_documents(handbook_chunks, ids=[f"cs999#{i}" for i in range(len(handbook_chunks))])

for collection in store._client.list_collections():
    name = getattr(collection, "name", collection)  # 💡 newer Chroma versions return names, older ones objects
    print(f"📚 {name:<12} {store._client.get_collection(name).count():>3} chunks")

## Part 3: Metadata filters

Metadata filters are exact, fast and **independent of embeddings**, so this part is fully meaningful even offline. Chroma's `where` uses a small Mongo-like language: `{"unit": 3}`, `{"page": {"$gte": 3}}`, `{"source": {"$in": [...]}}`, combined with `$and` / `$or`.

In [ ]:
for unit in (0, 1, 2, 3):
    found = store.get(where={"unit": unit}, include=["metadatas"])
    sources = {m["source"] for m in found["metadatas"]}
    print(f"unit {unit}: {len(found['ids']):>3} chunks from {sources}")

Filters combine with similarity search: Chroma finds the nearest neighbours **among the chunks that match**. Even with random vectors, every result is guaranteed to come from Unit 3:

In [ ]:
for doc in store.similarity_search("How do transactions stay consistent?", k=4, filter={"unit": 3}):
    print(f"unit {doc.metadata['unit']}  p{doc.metadata['page']}  §{doc.metadata['section']}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Now we filter on two things at once: Unit 2, pages 3 and later. The obvious way to write that is <code>{"unit": 2, "page": {"$gte": 3}}</code>. Predict: will Chroma accept it?</span>
        </td>
    </tr>
</table>

In [ ]:
try:
    store.get(where={"unit": 2, "page": {"$gte": 3}})
    print("Accepted")
except Exception as exc:  # 💡 Chroma raises a ValueError for this
    print(f"❌ {type(exc).__name__}: {str(exc)[:160]}")

Chroma allows only **one key at the top level** of `where`; several conditions must be wrapped in `$and`. GradeWise's retrievers do that for you with a tiny helper, so callers can write the natural form:

In [ ]:
from gradewise.rag.retrievers import to_chroma_where

where = to_chroma_where({"unit": 2, "page": {"$gte": 3}})
print("Rewritten filter:", where)

found = store.get(where=where, include=["metadatas"])
print(f"{len(found['ids'])} chunks, pages {sorted({m['page'] for m in found['metadatas']})}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">Metadata filters are how real products enforce <b>who may see what</b>: a tenant ID, a course code, a document's access level. They're exact (no "probably the right course") and they cost almost nothing. In GradeWise, a question in the networking unit can be told to search Unit 2 only, and in week 6 each course's knowledge base stays completely separate.</span>
        </td>
    </tr>
</table>

## Part 4: Idempotent ingestion

Embedding costs money and time. If Emma fixes a typo in Unit 3, re-embedding all four PDFs is wasteful; if nothing changed, re-embedding anything is pure waste. And if a re-run *added* the chunks again, every search would return each chunk twice.

**Idempotent** means "running it twice has the same effect as running it once". GradeWise's `ingest()` gets there with two ideas:

1. **A fingerprint per file.** A SHA-256 hash of the PDF's bytes. Same bytes, same hash; change one byte, completely different hash.
2. **Deterministic chunk IDs**, built from the file name, its hash and the chunk number, so re-adding a chunk overwrites it instead of duplicating it.

In [ ]:
from gradewise.ingestion.pipeline import file_hash

print(hashlib.sha256(b"A deadlock needs four conditions.").hexdigest())
print(hashlib.sha256(b"A deadlock needs four conditions!").hexdigest(), "← one character changed\n")

for pdf in datasets.kb_files(KB):
    print(f"{pdf.name:<30} {file_hash(pdf)[:16]}…")

`ingest()` stores each file's hash in its chunks' metadata. On the next run it compares: **unchanged** → skip, **changed** → delete the old chunks, then add the new ones, **new** → add, **gone from the folder** → delete.

![Diagram](../assets/diagrams/030d975c86f3.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
    P[📄 each PDF] --> H{hash in store?}
    H -->|same hash| S[⏭️ skip: $0]
    H -->|different hash| U[🗑️ delete old chunks → add new]
    H -->|not there| N[➕ add]
    X[🧹 in store but not in folder] --> D[🗑️ delete]
```

</details>


Let's run it for real (with fake embeddings, into a fresh temp folder). Run 1 ingests everything:

In [ ]:
from gradewise.ingestion import ingest

DB = TMP / "ingest_demo"
first = ingest(kb_dir=KB, collection="cs201", embeddings=fake_embeddings, persist_dir=DB)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Now we run exactly the same command again. Predict the numbers: how many chunks will be added, how many will be in the store, and how many PDFs skipped?</span>
        </td>
    </tr>
</table>

In [ ]:
second = ingest(kb_dir=KB, collection="cs201", embeddings=fake_embeddings, persist_dir=DB)
print(f"Run 1 added {first.chunks_added}, run 2 added {second.chunks_added}. Store holds {second.chunks_total}.")

🎉 **0 chunks added, 0 embedding calls, $0.** That's idempotency. Now let's test the other two branches on a *copy* of the knowledge base: we'll change one PDF (appending a few bytes changes its hash without breaking it) and delete another.

In [ ]:
kb_copy = TMP / "kb_copy"
shutil.copytree(KB, kb_copy, dirs_exist_ok=True)
db_copy = TMP / "ingest_copy"
ingest(kb_dir=kb_copy, collection="cs201", embeddings=fake_embeddings, persist_dir=db_copy, quiet=True)

unit1 = kb_copy / "unit1_operating_systems.pdf"  # 💡 unit1 and the handbook are in both DATA_SIZEs
unit1.write_bytes(unit1.read_bytes() + b"\n%edited by Emma\n")  # 💡 new hash, same content
(kb_copy / "cs201_course_handbook.pdf").unlink()  # 🧹 the handbook is withdrawn

third = ingest(kb_dir=kb_copy, collection="cs201", embeddings=fake_embeddings, persist_dir=db_copy, quiet=True)
print("skipped :", third.skipped)
print("updated :", third.updated)
print("removed :", third.removed)
print("chunks added this run:", third.chunks_added, "· chunks in store:", third.chunks_total)

Unit 1 was re-indexed (its old chunks deleted first, so no duplicates), the handbook's chunks were removed, and the unchanged units were never touched.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">Open <code>gradewise/ingestion/store.py</code> (<code>get_vectorstore</code>: the folder, the cosine metric, the injectable embeddings) and <code>gradewise/ingestion/pipeline.py</code> (<code>ingest</code>, <code>file_hash</code>, <code>IngestReport</code>). Read the ⚠️ comment about deleting <i>before</i> adding: it's the bug most home-made pipelines have. The real command, with real embeddings, is <code>uv run week2/pro_implementation/ingest.py</code>, and we'll run it properly on Day 5.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

Today you:
- 🕸️ saw why brute force doesn't scale and how HNSW's layered graph makes search fast, and chose cosine distance
- 🗄️ built a persistent Chroma store, reopened it from disk, and kept a separate collection per course
- 🔎 filtered by metadata, met Chroma's "one top-level key" rule and `to_chroma_where()`
- ♻️ ran `ingest()` twice (**0 chunks added** the second time), then watched it re-index a changed file and remove a deleted one

**Tomorrow (Day 4):** retrieval strategies. Dense search (what Chroma does), BM25 keyword search (which works offline, on real meaning-free maths), and hybrid search with reciprocal-rank fusion. Then your first RAG chain that answers with citations.

⏭ **Next:** `week2/day4.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">(1) Write a <code>where</code> filter for "chunks from Unit 1 <b>or</b> Unit 3, on page 4 or later" and check the pages and units that come back. (2) Prove that adding the same chunks again with the same IDs doesn't change the count, but adding them with <i>random</i> IDs does. (3) Make a real content change to a copy of a PDF (hint: open it with <code>pymupdf</code>, add a line of text to a page with <code>page.insert_text</code>, save it) and check <code>ingest()</code> reports it as updated. Solutions are in <code>week2/solutions/day3_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• Chroma docs: <a href="https://docs.trychroma.com/">docs.trychroma.com</a> (see "Metadata filtering")<br/> • The HNSW paper (Malkov & Yashunin): <a href="https://arxiv.org/abs/1603.09320">arxiv.org/abs/1603.09320</a><br/> • LangChain + Chroma: <a href="https://docs.langchain.com/oss/python/integrations/vectorstores/chroma">docs.langchain.com/oss/python/integrations/vectorstores/chroma</a></span>
        </td>
    </tr>
</table>